In [1]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns, glob, os
import scipy.io as sio

## Objectives
1. Create preQC_df with extra columns, do QC offline
2. Load QC; save keeps, maybes, questionables, mergers, notes neur figs in the data/units dir
3. Create neur_df with columns for spikes, region, coordinates
4. Merge clusters as needed
5. Inspect & save

### variables

In [2]:
# suffix = '_max'
suffix = ''

patient = 22
save = False

units_dir = f'../../data/units'

units_all_dir = f'{units_dir}/all/2025{patient}{suffix}'
units_waves_dir = f'{units_dir}/waves/2025{patient}{suffix}'

units_keeps_dir = f'{units_dir}/keeps/2025{patient}'
units_maybes_dir = f'{units_dir}/maybes/2025{patient}'
units_questionables_dir = f'{units_dir}/questionables/2025{patient}'
units_mergers_dir = f'{units_dir}/mergers/2025{patient}'
for dir in [units_all_dir, units_waves_dir,
            units_keeps_dir, units_maybes_dir, units_questionables_dir, units_mergers_dir]:
    os.makedirs(dir, exist_ok=True)

In [3]:
print(len(glob.glob(f'{units_all_dir}/*')), 'ALL units')

87 ALL units


### 1. set up preQC, populate only-cluster and only-wave fig directories

In [4]:
# save = True

# parse osort figs
for file in glob.glob(f'../../results/2025{patient}/osort_mat/figs{suffix}/5/*'):

    # grab individual CL figs
    if 'CL' in os.path.basename(file) and 'ALL' not in os.path.basename(file):
        
        dest = os.path.join(units_all_dir, os.path.basename(file))
        if not os.path.exists(dest):
            print(f'Copying to {dest}')
            if save: os.system(f'cp {file} {dest}')

    # grab WAVES figs to check for mergers
    if 'WAVES' in os.path.basename(file):
        
        dest = os.path.join(units_waves_dir, os.path.basename(file))
        if not os.path.exists(dest):
            print(f'Copying to {dest}')
            if save: os.system(f'cp {file} {dest}')

save = False # reset to be safe

### create preQC_df with extra columns

In [5]:
# init preQC_df 
preQC_df = pd.DataFrame(columns=['chanID', 'unitID'])
chanIDs, unitIDs = [], []

# parse units_all_dir
for file in glob.glob(f'{units_all_dir}/*'):
    chanIDs.append(int(os.path.basename(file).split('_')[0][1:]))
    unitIDs.append(int(os.path.basename(file).split('_')[2]))

# create df
preQC_df = pd.DataFrame({'chanID': chanIDs, 'unitID': unitIDs})
preQC_df = preQC_df.sort_values(by=['chanID', 'unitID']).reset_index(drop=True)

# create empty cols for keep, maybe, questionable, and mergers
preQC_df['keeps'], preQC_df['maybes'], preQC_df['questionables'], preQC_df['mergers'], preQC_df['notes'] = np.nan, np.nan, np.nan, np.nan, np.nan

# save df
preQC_df_path = f'../../results/2025{patient}/records/preQC_pt{patient}.csv'
print(f'Saving preQC_df to {preQC_df_path}')
preQC_df.to_csv(preQC_df_path, index=False)

save = False # reset to be safe
preQC_df

Saving preQC_df to ../../results/202522/records/preQC_pt22.csv


,chanID,unitID,keeps,maybes,questionables,mergers,notes
0,193,31,NaN,NaN,NaN,NaN,NaN
1,193,810,NaN,NaN,NaN,NaN,NaN
2,193,883,NaN,NaN,NaN,NaN,NaN
3,193,902,NaN,NaN,NaN,NaN,NaN
4,193,929,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...
82,208,1284,NaN,NaN,NaN,NaN,NaN
83,208,1301,NaN,NaN,NaN,NaN,NaN
84,208,1329,NaN,NaN,NaN,NaN,NaN
85,208,1336,NaN,NaN,NaN,NaN,NaN


### 2. load QC; save keeps/maybes/questionables/mergers neur figs in the data/units dir

In [6]:
save=True

QC_df = pd.read_csv(f'../../results/2025{patient}/records/QC_pt{patient}.csv')
keeps_df = QC_df[QC_df['keeps'] == 1].copy().reset_index(drop=True)
maybes_df = QC_df[~QC_df['maybes'].isna()].copy().reset_index(drop=True)
questionables_df = QC_df[QC_df['questionables'] == 1].copy().reset_index(drop=True)
mergers_df = QC_df[~QC_df['mergers'].isna()].copy().reset_index(drop=True)

# keeps + mergers
possible_neurs_df = pd.concat([keeps_df, mergers_df]).drop_duplicates().reset_index(drop=True)

for unit_file in glob.glob(f'{units_all_dir}/*'):
    chanID = int(os.path.basename(unit_file).split('_')[0][1:])
    unitID = int(os.path.basename(unit_file).split('_')[2])

    # keeps
    if ((keeps_df['chanID'] == chanID) & (keeps_df['unitID'] == unitID)).any():
        dest = f'{units_keeps_dir}/{os.path.basename(unit_file)}'
        if not os.path.exists(dest):
            print(f'Copying to {dest}')
            if save: os.system(f'cp {unit_file} {dest}')
    # maybes
    if ((maybes_df['chanID'] == chanID) & (maybes_df['unitID'] == unitID)).any():
        dest = f'{units_maybes_dir}/{os.path.basename(unit_file)}'
        if not os.path.exists(dest):
            print(f'Copying to {dest}')
            if save: os.system(f'cp {unit_file} {dest}')
    # questionables
    if ((questionables_df['chanID'] == chanID) & (questionables_df['unitID'] == unitID)).any():
        dest = f'{units_questionables_dir}/{os.path.basename(unit_file)}'
        if not os.path.exists(dest):
            print(f'Copying to {dest}')
            if save: os.system(f'cp {unit_file} {dest}')
    # mergers
    if ((mergers_df['chanID'] == chanID) & (mergers_df['unitID'] == unitID)).any():
        dest = f'{units_mergers_dir}/{os.path.basename(unit_file)}'
        if not os.path.exists(dest):
            print(f'Copying to {dest}')
            if save: os.system(f'cp {unit_file} {dest}')


for label, df, target_dir in [('keeps', keeps_df, units_keeps_dir),
                               ('maybes', maybes_df, units_maybes_dir),
                               ('mergers', mergers_df, units_mergers_dir)]:
    dir_files = set(os.path.basename(f) for f in glob.glob(f'{target_dir}/*.png'))
    df_files = set()
    for _, row in df.iterrows():
        matches = glob.glob(f'{units_all_dir}/A{int(row["chanID"])}_*_{int(row["unitID"])}_*.png')
        df_files.update(os.path.basename(f) for f in matches)
    extra = dir_files - df_files
    missing = df_files - dir_files
    if extra:   print(f'[{label}] extra files (in dir, not in df): {extra}')
    if missing: print(f'[{label}] missing files (in df, not in dir): {missing}')


assert len(keeps_df) == len(glob.glob(f'{units_keeps_dir}/*.png')), f'Length mismatch: keeps_df has {len(keeps_df)} rows, but {len(glob.glob(f"{units_keeps_dir}/*.png"))} files in {units_keeps_dir}'
assert len(maybes_df) == len(glob.glob(f'{units_maybes_dir}/*.png')), f'Length mismatch: maybes_df has {len(maybes_df)} rows, but {len(glob.glob(f"{units_maybes_dir}/*.png"))} files in {units_maybes_dir}'
# assert len(mergers_df) == len(glob.glob(f'{units_mergers_dir}/*.png')), f'Length mismatch: mergers_df has {len(mergers_df)} rows, but {len(glob.glob(f"{units_mergers_dir}/*.png"))} files in {units_mergers_dir}'

save = False # reset to be safe
possible_neurs_df

[mergers] extra files (in dir, not in df): {'A199_WAVES_1_THM_1.png'}


,chanID,unitID,keeps,maybes,questionables,mergers
0,193,883,1.0,NaN,NaN,NaN
1,193,929,1.0,NaN,NaN,NaN
2,200,889,1.0,NaN,NaN,NaN
3,201,2271,1.0,NaN,1.0,NaN
4,201,2283,1.0,NaN,1.0,NaN
5,204,1726,1.0,NaN,1.0,NaN
6,199,2436,NaN,FR fluctuation,NaN,2.0
7,199,2472,NaN,FR fluctuation,NaN,2.0


### 3. create neur_df for neurs in possible_neurs_df and taking in spike times from osort mat files

### helpers

In [7]:
def getunitID2spikes(unitIDs, spikes, possible_neurs_df):
    ''' return dict with keys=unique units, and vals = list of corresponding spikes '''
    
    # initialize unit2spikes with keys as unique unit IDs and empty list as values
    unit2spikes = {}
    for unitID, spike in zip(unitIDs, spikes):

        if unitID not in possible_neurs_df['unitID'].tolist(): continue
        if unitID not in unit2spikes: unit2spikes[unitID] = [] # initialize

        unit2spikes[unitID].append(spike)

    return unit2spikes

### First, add spike data from OSort mats.

In [8]:
samp_rate = 1000000
# columns
chanID_list, unitID_list, spikes_list, num_spikes_list, FR_list = [], [], [], [], []

# go through OSort mat files
for mat_file in glob.glob(f'../../results/2025{patient}/osort_mat/sorted_mats/5/*_sorted_new.mat'):

    # load channel mat
    chanMat = sio.loadmat(mat_file)
    chanID = int(os.path.basename(mat_file).split('_')[0][1:])

    # load vectors of spike times and corresponding units
    if chanMat['assignedNegative'].size == 0: continue
    spike_vector = chanMat['newTimestampsNegative'][0] # len = total n_spikes
    unit_vector = chanMat['assignedNegative'][0] # len = total n_spikes

    # create unit_vector => [spike_vector] for QCed units
    unit2spikes = getunitID2spikes(unit_vector, spike_vector, possible_neurs_df)
    for unitID, unit_spike_list in unit2spikes.items():
        chanID_list.append(chanID)
        unitID_list.append(unitID)
        spikes = np.array(unit_spike_list) / samp_rate  # seconds
        spikes_list.append(spikes)
        num_spikes_list.append(len(spikes))
        FR_list.append(len(spikes) / (spikes[-1] - spikes[0]))

neur_df = pd.DataFrame({'chanID': chanID_list, 'unitID': unitID_list, 'spikes': spikes_list, 'num_spikes': num_spikes_list, 'FR': FR_list})
neur_df = pd.merge(
    neur_df,
    possible_neurs_df[['chanID', 'unitID', 'keeps', 'mergers']],
    on=['chanID', 'unitID'],
    how='left'
)
neur_df = neur_df.sort_values(by=['chanID', 'unitID']).reset_index(drop=True)
assert len(neur_df) == len(possible_neurs_df), f'Length mismatch: neur_df has {len(neur_df)} rows, possible_neurs_df has {len(possible_neurs_df)} rows'
neur_df


,chanID,unitID,spikes,num_spikes,FR,keeps,mergers
0,193,883,"[3.3423000000000003, 4.025933333333334, 4.1788...",2431,1.446764,1.0,NaN
1,193,929,"[1.4766333333333335, 1.5566666666666666, 1.716...",7246,4.303382,1.0,NaN
2,199,2436,"[6.560433333333334, 12.252566666666668, 47.828...",2996,1.784500,NaN,2.0
3,199,2472,"[7.637333333333334, 16.9942, 40.48683333333334...",1177,0.702765,NaN,2.0
4,200,889,"[1.2737333333333334, 4.168333333333334, 5.4483...",3236,1.921668,1.0,NaN
5,201,2271,"[17.292166666666667, 17.33396666666667, 18.380...",4520,2.710302,1.0,NaN
6,201,2283,"[18.104666666666667, 25.1578, 32.396, 47.1109,...",2065,1.238960,1.0,NaN
7,204,1726,"[17.384533333333337, 17.67366666666667, 17.679...",2360,1.420181,1.0,NaN


### add region column by mapping channel -> region (label)

In [9]:
if patient != 22:
    chanMap = sio.loadmat(glob.glob(f'../../results/2025{patient}/records/*ChannelMap*.mat')[0])

    # variable across patients
    if patient in [12, 21]: channelMap = chanMap['ChannelMap1'].flatten()
    elif patient == 18: channelMap = chanMap['ChannelMap2'].flatten()

    labelMap = chanMap['LabelMap'].flatten() # contains region labels
    labelMap = np.array([str(label.squeeze()) for label in labelMap])  # convert to str

    nan_mask = ~np.isnan(channelMap)
    channel2label = dict(zip(channelMap[nan_mask], labelMap[nan_mask]))

    neur_df['region'] = neur_df['chanID'].map(channel2label).fillna(neur_df['chanID']).apply(lambda x: str(x))

else: neur_df['region'] = np.arange(len( neur_df)) # for pt22, just assign region = chanID since no chanMap available

neur_df


,chanID,unitID,spikes,num_spikes,FR,keeps,mergers,region
0,193,883,"[3.3423000000000003, 4.025933333333334, 4.1788...",2431,1.446764,1.0,NaN,0
1,193,929,"[1.4766333333333335, 1.5566666666666666, 1.716...",7246,4.303382,1.0,NaN,1
2,199,2436,"[6.560433333333334, 12.252566666666668, 47.828...",2996,1.784500,NaN,2.0,2
3,199,2472,"[7.637333333333334, 16.9942, 40.48683333333334...",1177,0.702765,NaN,2.0,3
4,200,889,"[1.2737333333333334, 4.168333333333334, 5.4483...",3236,1.921668,1.0,NaN,4
5,201,2271,"[17.292166666666667, 17.33396666666667, 18.380...",4520,2.710302,1.0,NaN,5
6,201,2283,"[18.104666666666667, 25.1578, 32.396, 47.1109,...",2065,1.238960,1.0,NaN,6
7,204,1726,"[17.384533333333337, 17.67366666666667, 17.679...",2360,1.420181,1.0,NaN,7


### add coordinates columns by mapping region->coords

In [10]:
# cleaning function to handle nested arrays and bytes
def clean_entry(x):
    while isinstance(x, (np.ndarray, list)):
        x = x[0]
    if isinstance(x, (bytes, bytearray)):
        x = x.decode("utf-8", errors="ignore")
    return str(x)

In [11]:
# load
electrodeInfo = sio.loadmat(glob.glob(
                f'../../results/2025{patient}/records/*DI_Electrodes*.mat')[0])
ElecMapRaw   = pd.DataFrame(electrodeInfo['ElecMapRaw']) # region -> ID
ElecXYZRaw   = pd.DataFrame(electrodeInfo['ElecXYZRaw']) # ID -> coordinates
# ElecAtlasRaw = pd.DataFrame(electrodeInfo['ElecAtlasRaw']) # atlas coords?
# atlas_index = 0 # NMM

# clean
region_s = ElecMapRaw[0].apply(clean_entry)                    # Series of regions
# atlas_s  = ElecAtlasRaw.iloc[:, atlas_index].apply(clean_entry)  # Series of atlas regions

# build small tables
region2id_df = pd.DataFrame({"region": region_s.values,
                             "ID": np.arange(len(region_s))})
id2xyz_df = ElecXYZRaw.reset_index().rename(columns={'index':'ID', 0:'x', 1:'y', 2:'z'})
# xyz2atlasRegions = pd.DataFrame({"ID": np.arange(len(atlas_s)),
#                                  "atlas_region": atlas_s.values})

# merge
if patient == 22:
    neur_df = neur_df.copy()
    neur_df[['x', 'y', 'z']] = 0
else:
    neur_df = (neur_df
                    .merge(region2id_df, on='region', how='left')
                    .merge(id2xyz_df, on='ID', how='left'))
                  #   .merge(xyz2atlasRegions, on='ID', how='left')
    neur_df = neur_df.drop(columns=['ID'])
    
# sort by chanID
neur_df = neur_df.sort_values(by=['chanID', 'unitID']).reset_index(drop=True)
neur_df

,chanID,unitID,spikes,num_spikes,FR,keeps,mergers,region,x,y,z
0,193,883,"[3.3423000000000003, 4.025933333333334, 4.1788...",2431,1.446764,1.0,NaN,0,0,0,0
1,193,929,"[1.4766333333333335, 1.5566666666666666, 1.716...",7246,4.303382,1.0,NaN,1,0,0,0
2,199,2436,"[6.560433333333334, 12.252566666666668, 47.828...",2996,1.784500,NaN,2.0,2,0,0,0
3,199,2472,"[7.637333333333334, 16.9942, 40.48683333333334...",1177,0.702765,NaN,2.0,3,0,0,0
4,200,889,"[1.2737333333333334, 4.168333333333334, 5.4483...",3236,1.921668,1.0,NaN,4,0,0,0
5,201,2271,"[17.292166666666667, 17.33396666666667, 18.380...",4520,2.710302,1.0,NaN,5,0,0,0
6,201,2283,"[18.104666666666667, 25.1578, 32.396, 47.1109,...",2065,1.238960,1.0,NaN,6,0,0,0
7,204,1726,"[17.384533333333337, 17.67366666666667, 17.679...",2360,1.420181,1.0,NaN,7,0,0,0


### 4. Merge clusters as needed

In [12]:
merged_rows = []
merge_mask = neur_df['mergers'].notna()

# extract merger rows as df
for _, merge_group in neur_df[merge_mask].groupby('mergers'):
    merged_spikes = np.sort(np.concatenate(merge_group['spikes'].values))

    # update row with merged info
    first_row = merge_group.iloc[0]
    merged_rows.append({
        **{col: first_row[col] for col in neur_df.columns},
        'unitID':     '/'.join(merge_group['unitID'].astype(str).tolist()),
        'spikes':     merged_spikes,
        'num_spikes': len(merged_spikes),
        'FR':         len(merged_spikes) / (merged_spikes[-1] - merged_spikes[0]),
        'keeps':      1,
        # chanID, region, merge_cluster, x, y, z already from iloc[0]
    })

# concat original df without merged rows + new merged rows
neur_df = pd.concat(
    [neur_df[~merge_mask], pd.DataFrame(merged_rows)],
    ignore_index=True
)
neur_df = neur_df.sort_values(by=['chanID']).reset_index(drop=True)
neur_df

,chanID,unitID,spikes,num_spikes,FR,keeps,mergers,region,x,y,z
0,193,883,"[3.3423000000000003, 4.025933333333334, 4.1788...",2431,1.446764,1.0,NaN,0,0,0,0
1,193,929,"[1.4766333333333335, 1.5566666666666666, 1.716...",7246,4.303382,1.0,NaN,1,0,0,0
2,199,2436/2472,"[6.560433333333334, 7.637333333333334, 12.2525...",4173,2.485553,1.0,2.0,2,0,0,0
3,200,889,"[1.2737333333333334, 4.168333333333334, 5.4483...",3236,1.921668,1.0,NaN,4,0,0,0
4,201,2271,"[17.292166666666667, 17.33396666666667, 18.380...",4520,2.710302,1.0,NaN,5,0,0,0
5,201,2283,"[18.104666666666667, 25.1578, 32.396, 47.1109,...",2065,1.238960,1.0,NaN,6,0,0,0
6,204,1726,"[17.384533333333337, 17.67366666666667, 17.679...",2360,1.420181,1.0,NaN,7,0,0,0


### 5. inspect, typecast, & save

In [ ]:
neur_df['spikes'] = neur_df['spikes'].apply(lambda x: np.array(x))  # ensure arrays
print('example neuron')
print(f'last 5 spikes (s): {neur_df["spikes"].iloc[0][-5:]}')
print(f'last 5 spikes (min): {neur_df["spikes"].iloc[0][-5:]/60}')

neur_df['unitID'] = neur_df['unitID'].astype(str)
 
# add patient as first column
if 'patient' not in neur_df: neur_df.insert(0, 'patient', patient)

# make dir
processed_data_dir = f'../../results/2025{patient}/records/processed_data'
os.makedirs(processed_data_dir, exist_ok=True)

# save file
parquet_path = os.path.join(processed_data_dir, 'df_neurs.parquet')
if not os.path.exists(parquet_path):
    print('saving neur_df')
    neur_df.to_parquet(parquet_path, index=False)
# neur_df.to_parquet(parquet_path, index=False)


example neuron
last 5 spikes (s): [1679.8615     1680.24126667 1680.958      1683.08876667 1683.64356667]
last 5 spikes (min): [27.99769167 28.00402111 28.01596667 28.05147944 28.06072611]
